# Primate Vocalization Detection — Colab Run

End-to-end pipeline: **mount Drive → clone latest code → load data → train VGG19 → detect on long audio → visualize**.

## Before you start

1. `Runtime` → `Change runtime type` → **Hardware accelerator: T4 GPU**.
2. Run cells **in order, one at a time**.
3. If any cell fails, **stop and fix it** before moving on.
4. Your data must already be at `My Drive/primates-data/` with subfolders `species/`, `background/`, `long_audio/`.

## Step 0 — Confirm GPU is available

If this prints `GPUs detected: []` you forgot to switch the runtime. Fix it first.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs detected:      {gpus}')
if not gpus:
    print('\nWARNING: No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')
else:
    print('\nOK: GPU ready.')

## Step 1 — Mount Google Drive

You'll be prompted to authorize. After this `/content/drive/MyDrive` should map to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Verify the data layout

This cell does **not** modify anything. It just walks your `primates-data/` folder and prints how many `.wav` files it can see in each subfolder. **Stop here if any species or background folder shows 0 files** — that means the folder name in `config.py` does not match what's actually on Drive.

In [ ]:
import os

DATA_ROOT = '/content/drive/MyDrive/primates-data'
assert os.path.isdir(DATA_ROOT), f'Cannot find {DATA_ROOT} - check the path in My Drive.'

print(f'DATA_ROOT = {DATA_ROOT}\n')

for sub in ['species', 'background', 'long_audio']:
    p = os.path.join(DATA_ROOT, sub)
    if not os.path.isdir(p):
        print(f'MISSING  {sub}/  (expected at {p})')
        continue
    children = sorted(c for c in os.listdir(p) if not c.startswith('.'))
    print(f'OK  {sub}/  ({len(children)} entries)')
    for c in children:
        full = os.path.join(p, c)
        if os.path.isdir(full):
            n_wav = sum(1 for _, _, fs in os.walk(full) for f in fs if f.lower().endswith('.wav'))
            print(f'      {c:50s} -> {n_wav} .wav files')
        elif c.lower().endswith('.wav'):
            print(f'      {c}')

## Step 3 — Clone the latest source code from GitHub

We clone the repository into `/content/primates-sound-detection`. This guarantees you're running the latest code regardless of what's in the `src/` folder on your Drive.

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!git log --oneline -5

## Step 4 — Install Python dependencies

Colab already has TensorFlow / NumPy / pandas / matplotlib. This mainly installs librosa, soundfile and opencv-python.

In [ ]:
!pip install -q -r requirements.txt
print('\nDone installing requirements.')

## Step 5 — Configure paths and load the project config

We:
1. Add `src/` to `sys.path` so the modules can be imported as `import config`, `import train`, etc.
2. Set `PRIMATE_DATA_ROOT` so `config.py` knows where your Drive data lives.
3. Print the resulting configuration as a final sanity check.

In [ ]:
import os, sys, importlib

PROJECT_ROOT = '/content/primates-sound-detection'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ['PRIMATE_DATA_ROOT']     = '/content/drive/MyDrive/primates-data'
os.environ['PRIMATE_MODEL_POOLING'] = 'temporal_freqpos'  # production V11 head (default is 'gap')

import config
importlib.reload(config)  # re-read env vars in case a stale copy was cached
config.print_config_summary()

## Step 6 — Data sanity check (load species + background)

**This is the most important checkpoint.** It actually loads every `.wav` file off Drive into memory and prints how many samples per class. If a species shows 0 samples, the path in `config.SPECIES_FOLDERS` does not match your Drive.

Drive I/O is slow the first time — this can take a few minutes for the full dataset. Be patient.

In [ ]:
import data_loader
import importlib; importlib.reload(data_loader)

species_data = data_loader.load_species_data()
background_data = data_loader.load_background_data()
data_loader.print_data_summary(species_data, background_data)

total_species = sum(len(v) for v in species_data.values())
assert total_species > 0, 'No species samples loaded - check the folder names in src/config.py against your Drive!'
assert len(background_data) > 0, 'No background samples loaded - check the folder names in src/config.py against your Drive!'
print(f'\nSanity check passed: {total_species} species samples, {len(background_data)} background samples.')

## Step 7 — Load the trained model (or retrain from scratch)

By default this cell **loads** the previously-trained model from `outputs/models/best_model_v12.h5` (falling back to `best_model.h5`) on your Drive — which is what you want most of the time, since training takes 20–60 min on a T4.

Flip `FORCE_RETRAIN = True` if you actually want to retrain (e.g. you added new species data, changed the augmentation config, or the saved model is missing). Retraining runs the full pipeline: re-load audio → mel-spectrograms → augmentation → VGG19 + head → up to 50 epochs with early stopping → save best model → print metrics & confusion matrix.

In [ ]:
FORCE_RETRAIN = False  # set to True to retrain from scratch

import model as model_module
import importlib; importlib.reload(model_module)

candidates = ['best_model_v12.h5', 'best_model.h5']
best_model_path = next(
    (os.path.join(config.MODEL_SAVE_DIR, n)
     for n in candidates if os.path.exists(os.path.join(config.MODEL_SAVE_DIR, n))),
    os.path.join(config.MODEL_SAVE_DIR, candidates[0]))

if FORCE_RETRAIN or not os.path.exists(best_model_path):
    if FORCE_RETRAIN:
        print('FORCE_RETRAIN=True — running full training pipeline.')
    else:
        print(f'No saved model at {best_model_path} — running full training pipeline.')
    import train
    importlib.reload(train)
    model = train.run_complete_training_pipeline()
    print('\nTraining done. Best model saved to:', best_model_path)
else:
    print(f'Loading existing model from: {best_model_path}')
    model = model_module.load_trained_model(best_model_path)
    model_module.print_model_summary(model)
    print('\nModel ready. (Set FORCE_RETRAIN=True above if you want to retrain.)')


## Step 8 — Run sliding-window inference on the long recordings (ONCE)

For every file under `primates-data/long_audio/` this slides a 5 s window with 2.5 s stride and runs the trained model on each window. We do this **once** and keep the raw softmax scores in memory so the next step can compare several confidence thresholds without paying for another full forward pass.

This is the slow cell — expect ~5–15 min per hour of audio on a T4. If you OOM, drop `config.BATCH_SIZE` in Step 5 and re-run.

In [ ]:
import detection
import importlib; importlib.reload(detection)

raw_results = detection.run_raw_inference_all(model)

total_windows = sum(len(t) for t, _ in raw_results.values())
print(f'\nRaw inference done: {total_windows} windows across {len(raw_results)} files.')

## Step 8a — Threshold sweep: how many detections at each cutoff?

The default `DETECTION_CONFIDENCE_THRESHOLD = 0.7` was set very conservatively. On real field recordings (where calls are off-centre, farther from the mic, and noisier than training clips) 0.7 routinely misses most vocalisations.

This cell re-applies several thresholds to the **same** raw predictions, applies NMS at each, and shows how many detections per species survive. Use the chart to pick a threshold that gives you a manageable number of clips to listen through without cratering precision.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

THRESHOLDS = [0.7, 0.5, 0.4, 0.3, 0.25, 0.2]
sweep = detection.sweep_thresholds(raw_results, THRESHOLDS)

# Build a tidy summary: threshold x species -> count
rows = []
for thr, per_file in sweep.items():
    per_species_counts = {}
    for df in per_file.values():
        if len(df) == 0:
            continue
        for sp, n in df['species'].value_counts().items():
            per_species_counts[sp] = per_species_counts.get(sp, 0) + int(n)
    rows.append({'threshold': thr, 'total': sum(per_species_counts.values()), **per_species_counts})

summary = pd.DataFrame(rows).fillna(0).sort_values('threshold', ascending=False)
species_cols = [c for c in summary.columns if c not in ('threshold', 'total')]
for c in species_cols + ['total']:
    summary[c] = summary[c].astype(int)
print('Detections surviving NMS at each threshold:')
print(summary.to_string(index=False))

# Bar chart stacked by species
fig, ax = plt.subplots(figsize=(10, 5))
bottom = [0] * len(summary)
x_labels = [f'{t:.2f}' for t in summary['threshold']]
for sp in species_cols:
    vals = summary[sp].tolist()
    ax.bar(x_labels, vals, bottom=bottom, label=sp)
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('Detections (after NMS)')
ax.set_title('Detections per species vs confidence threshold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATION_DIR, 'threshold_sweep.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"\nChart saved to {os.path.join(config.VISUALIZATION_DIR, 'threshold_sweep.png')}")

## Step 8b — Pick a threshold and build the final `all_detections`

Look at the sweep above and pick a value for `CHOSEN_THRESHOLD`. Good rule of thumb: pick the lowest threshold at which the **total** count is still small enough that you're willing to listen through every clip. If the model is working at all, you should see the count climb smoothly as the threshold drops.

This cell also saves one CSV per file under `outputs/detections/` at the chosen threshold so you have a permanent record.

In [ ]:
CHOSEN_THRESHOLD = 0.4  # <-- change this based on the sweep above

all_detections = sweep[CHOSEN_THRESHOLD] if CHOSEN_THRESHOLD in sweep else \
    detection.sweep_thresholds(raw_results, [CHOSEN_THRESHOLD])[CHOSEN_THRESHOLD]

total = 0
for fname, df in all_detections.items():
    detection.save_detections(df, fname)
    total += len(df)
print(f'\nChosen threshold {CHOSEN_THRESHOLD}: {total} detections across {len(all_detections)} files.')

# Quick peek
for fname, df in all_detections.items():
    if len(df) > 0:
        print(f'\n{fname}: {len(df)} detections (showing up to 10):')
        print(df.head(10)[['start_time', 'end_time', 'species', 'confidence']].to_string(index=False))
        break
else:
    print('\nNo detections at this threshold. Lower CHOSEN_THRESHOLD and re-run this cell.')

## Step 8c — Extract detected clips for manual listening review

For each detection above, this cuts a short WAV (with 0.5 s of padding on each side) out of the original long recording and saves it under `outputs/detected_clips/<species>/`. You can then open that folder in Drive and **listen to every detection by hand** to verify the model is right.

Filename layout: `<species>__<source>__<start_seconds>s__conf<score>.wav` so files sort naturally by species, then by time, with the model's confidence visible in the name.

In [ ]:
import utils
import importlib; importlib.reload(utils)

clips_dir = utils.extract_all_detected_clips(
    all_detections,
    padding=0.5,
    organize_by_species=True,
)
print(f'\nListen to clips at: {clips_dir}')

## Step 9 — Visualize and summarize

Renders waveform + spectrogram with detection overlays for each long-audio file, plus an overall stats block. PNGs are saved to `outputs/visualizations/`.

In [ ]:
import utils
import importlib; importlib.reload(utils)

utils.print_detection_statistics(all_detections)
utils.visualize_all_detections(all_detections)

print('\nAll outputs saved under /content/drive/MyDrive/primates-data/outputs/')
print('  - models/best_model_v12.h5')
print('  - detections/*.csv')
print('  - visualizations/*.png')

## Done

Your trained model, detection CSVs and visualizations are now persisted on Drive. To retrain after changing data or config, restart the runtime and re-run from Step 1.

If something failed: tell Claude the cell number and the error message — don't keep going.